In [ ]:
from dotenv import load_dotenv
from langchain_classic.agents.agent_toolkits import create_conversational_retrieval_agent
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever

load_dotenv()
from token import tok_name

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma, FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_text_splitters import CharacterTextSplitter

embedding = HuggingFaceEmbeddings()


def load_pdf_into_vectorstore(path: str):
    document_loader = PyPDFLoader(path)
    documents = document_loader.load()
    document_chunks = CharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separator="."
    ).split_documents(documents)
    persist_directory = "./faiss_db"
    vstore = FAISS.from_documents(documents=document_chunks, embedding=embedding)
    vstore.save_local(persist_directory)
    print("Uploaded document to vector store")
    return vstore


# load_pdf_into_vectorstore("Griffiths - Introduction to Quantum Mechanics 3rd ed 2018.pdf")

vector_store = FAISS.load_local("./faiss_db", embeddings=embedding, allow_dangerous_deserialization=True)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={'k': 10})


from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

llm = ChatOpenAI(model="gpt-5-nano")

# prompt = """
#     Retrieve the answers only from the context. if the context doesn't have anything related to the question. Answer "I don't know"
#
#     {input}
#
#     Context:
#         {context}
# """

# template = ChatPromptTemplate.from_messages([("human", prompt)])

# stuffed_chain = create_stuff_documents_chain(llm, template)
#
# chain = create_retrieval_chain(retriever, stuffed_chain)
#
# # chain = RunnableParallel({ "input": RunnablePassthrough(), "context": retriever}) | template | llm | StrOutputParser()
#
# chain.invoke({"input": "what is schrogingers equation ?"})

messages = ChatPromptTemplate.from_messages([
    ("system",
     '''Retrieve the answers only from the context. if the context doesn't have anything related to the question. Answer "I don't know"'''),
    MessagesPlaceholder(variable_name="chat-history"),
    ("human", "{input}, context: {context}")
])

refactor_prompt = '''
    Given the above chat history, rephrase the question
    with reference to the context. do not answer if the input
    contains a question. just rephrase the question. if the
    input doesn't match the question, return it as it is.

    {input}
'''

history_prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="chat-history"),
    ("human", refactor_prompt),
])

har = create_history_aware_retriever(llm, retriever, history_prompt)

stuffed_chain = create_stuff_documents_chain(llm, messages)

chain = create_retrieval_chain(har, stuffed_chain)

response = chain.invoke({
    'input': "can you list the chapter names ?",
    'chat-history': []
})

print( 'response -> ', response['answer'])


In [ ]:
chain.invoke({
    'input': " explain those topics with 2 bullet points",
    'chat-history': [
        ('human', "can you list the chapter names ?"),
        ('system', response['answer'])
    ]
})['answer']